# Build `Evaluation Results Database`

### `geometric` & `radiometric`

Paul Montesano, PhD  
June-Sept 2026

In [1]:
options(warn=-1)
library(tidyverse)
library(patchwork)
library(readr)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


```
CSDA_eval/results/evaluation_results_db
 └─ acquisitions.csv             # All acquisitions use across all CSDA optical evaluations
 └─ radiometric_db
   ├── results_snr.csv           # SNR metrics
   ├── results_abscal.csv        # absolute calibration accuracy metrics  
   └── results_tempstab.csv      # temporal stability metrics
 └─ geometric_db             
   ├── results_apa.csv           # abs position accuracy metrics
   ├── results_ssr.csv           # sensor spatial response metrics 
   ├── results_bbr.csv           # band-to-band ratio metrics  
   └── results_tempstab.csv      # temporal stability metrics

In [2]:
DIR_EVAL_RESULTS_DB = "/explore/nobackup/projects/CSDA_eval/results/evaluation_results_db"

### Find all csvs

In [3]:
# Define the root search path
search_path <- "/explore/nobackup/projects/CSDA_eval/results"

# 1. Grab ALL CSV files recursively down the tree
exclude_strings <- c('overview', 'smry', 'RSR','test','evaluation_results_db')  # Extend as needed
exclude_pattern <- paste(exclude_strings, collapse = "|")

all_csvs <- list.files(
  path = search_path,
  pattern = "\\.csv$",    # Regular expression matching strings that end with .csv
  recursive = TRUE,
  full.names = TRUE
) %>%
  .[!grepl(exclude_pattern, basename(.), ignore.case = TRUE)]  # ignore.case for robustness

print(glue::glue("Found {length(all_csvs)} CSV files"))
print(glue::glue("Excluded pattern: '{exclude_pattern}'"))

Found 64 CSV files
Excluded pattern: 'overview|smry|RSR|test|evaluation_results_db'


In [4]:
source('../lib/evallib.r')

In [5]:
# read_evaluation_files <- function(files_list){

#     ACQ_COLS <- c(
#       'evaluation_category','evaluation_type', 'vendor', 'affiliation', 'constellation', 'satellite',
#       'site_name', 'product_level',
#       'acq_datetime', 
#         #'date', 
#         'source_file'
#         )
    
#     # Band order defined inside function so it's always available
#     band_order <- c(
#         'Pan', 'Coastal', 'CB', 'Blue', 'B', 'Green', 'G', 'Yellow', 'Y', 
#         'Red', 'R', 'RedEdge', 'RE', 'RedEdge1', 'RE1', 'RedEdge2', 'RE2', 
#         'NIR', 'NIR1', 'NIR2', 'SWIR1', 'SWIR2'
#     )

#     df = files_list %>%
#             set_names() %>%
#             map_df(read_csv, .id = "source_file", show_col_types=FALSE) %>%
#             mutate(
#             evaluation_category = if_else(
#               grepl("radiometric", source_file, ignore.case = TRUE), 
#               "radiometric", 
#               "geometric"
#             )
#           ) %>%
            
#             # Drop empty unnamed trailing columns
#             select(-matches("^\\.\\.\\.\\d+$")) %>%
#             select(where(~ !all(is.na(.x)))) %>%
            
#             # Standardize column names
#             rename_with(~ case_when(
#                 .x == "Evaluation Type" ~ "Evaluation_type",
#                 TRUE                    ~ .x
#             )) %>%
#             rename_with(tolower) %>%
            
#             # # Add missing common acquisition cols as NA
#             # { if (!"Date_time_in_UTC(YYYYMMDD_HHMMSS)" %in% names(.))
#             #     mutate(., `Date_time_in_UTC(YYYYMMDD_HHMMSS)` = NA_character_) else . } %>%
#             # { if (!"Time_series_date_min(YYYYMMDD)" %in% names(.))
#             #     mutate(., `Time_series_date_min(YYYYMMDD)` = NA_character_) else . } %>%
#             # { if (!"Time_series_date_max(YYYYMMDD)" %in% names(.))
#             #     mutate(., `Time_series_date_max(YYYYMMDD)` = NA_character_) else . } %>%
#             { if (!"acq_datetime" %in% names(.))
#                 mutate(., acq_datetime = NA_character_) else . } %>%
#             { if (!"acq_id" %in% names(.))
#                 mutate(., acq_id = NA_character_) else . } %>%
#             { if (!"satellite" %in% names(.))
#                 mutate(., satellite = NA_character_) else . } %>%
#             { if (!"band_name" %in% names(.))
#                 mutate(., band_name = NA_character_) else . } %>%
             
#             { if ("vendor" %in% names(.))
#                 mutate(., affiliation = vendor) %>% select(-vendor) else . } %>%
            
#             mutate(
#                 # KEY FIX: convert acq_datetime to character first
#                 # so it's always a string going into parse_date_time
#                 # regardless of how read_csv interpreted it
#                 acq_datetime = case_when(
#                     !is.na(acq_datetime) ~
#                         parse_date_time(as.character(acq_datetime),
#                                        orders = c('YmdHMS', 'Ymd HMS',
#                                                   'Y-m-d H:M:S', 'Y-m-d',
#                                                   'Ymd',
#                                                   'mdy',        # ADD: handles 11/3/2024
#                                                   'mdY',        # ADD: handles 11/03/2024
#                                                   'dmy'),       # ADD: handles European format
#                                        tz = 'UTC'),
#                     !is.na(acq_datetime) ~
#                         parse_date_time(acq_datetime,
#                                        orders = 'YmdHMS',
#                                        tz = 'UTC'),
#                     TRUE ~ NA_POSIXct_
#                 ),
#                 affiliation = str_to_title(affiliation),
#                 constellation = str_to_title(constellation),
#                 .keep = "unused"
#             ) %>%
#             mutate(
#                 band_name = factor(band_name, levels = band_order),
#                 satellite = factor(satellite)
#             ) %>%
#             relocate(any_of(ACQ_COLS), .before = everything()) %>%
#             relocate(source_file, .after = last_col()) %>%
#             filter(if_any(-source_file, ~ !is.na(.))) # Removes extra rows that are just NA

#     return(df)
# }

### Radiometric results csvs

In [6]:
snr_files <-         grep("/radiometric/snr/.*_snr_eachcase\\.csv$", all_csvs, value = TRUE)
abscal_files <-      grep("/radiometric/abscal/.*_abscal_eachcase\\.csv$", all_csvs, value = TRUE)
tempstabrad_files <- grep("/radiometric/tempstab/.*_tempstab_summary.csv$", all_csvs, value = TRUE)

In [7]:
snr_files

[1] "/explore/nobackup/projects/CSDA_eval/results/airbus/pleiades/radiometric/snr/Radiometric_evaluation_Airbus_Pleiades_snr_eachcase.csv"      
[2] "/explore/nobackup/projects/CSDA_eval/results/airbus/pleiadesneo/radiometric/snr/Radiometric_evaluation_Airbus_PleiadesNeo_snr_eachcase.csv"
[3] "/explore/nobackup/projects/CSDA_eval/results/airbus/spot6/radiometric/snr/Radiometric_evaluation_Airbus_SPOT_snr_eachcase.csv"             
[4] "/explore/nobackup/projects/CSDA_eval/results/maxar/legion/radiometric/snr/Radiometric_evaluation_Vantor_Legion_snr_eachcase.csv"           
[5] "/explore/nobackup/projects/CSDA_eval/results/satellogic/newsat/radiometric/snr/Radiometric_evaluation_Satellogic_NewSat_snr_eachcase.csv"

In [8]:
# Usage - each file type read separately
df_snr      <- read_evaluation_files(snr_files)
df_abscal      <- read_evaluation_files(abscal_files)
df_tempstabrad      <- read_evaluation_files(tempstabrad_files)

In [9]:
colnames(df_tempstabrad)

[1] "evaluation_category"                                                                          
 [2] "evaluation_type"                                                                              
 [3] "affiliation"                                                                                  
 [4] "constellation"                                                                                
 [5] "satellite"                                                                                    
 [6] "site_name"                                                                                    
 [7] "product_level"                                                                                
 [8] "acq_datetime"                                                                                 
 [9] "number_of_images"                                                                             
[10] "acq_datetime_min"                                                                             
[11] "acq_datetime_max"                                                                             
[12] "time_series_length(days)"                                                                     
[13] "band_name"                                                                                    
[14] "radiometric_temporal_stability_toa_reflectance(%/year)"                                       
[15] "radiometric_temporal_stability_toa_reflectance_reference(%/year)"                             
[16] "radiometric_temporal_stability_toa_reflectance(coeff_of_variation%)"                          
[17] "radiometric_temporal_stability_toa_reflectance_reference(coeff_of_variation%)"                
[18] "radiometric_temporal_stability_gain_toa_reflectance(%/year)"                                  
[19] "radiometric_temporal_stability_gain_toa_reflectance(coeff_of_variation%)"                     
[20] "radiometric_temporal_stability_surface_reflectance(%/year)"                                   
[21] "radiometric_temporal_stability_surface_reflectance_reference_modisbrf(%/year)"                
[22] "radiometric_temporal_stability_surface_reflectance_reference_maiacac(%/year)"                 
[23] "radiometric_temporal_stability_surface_reflectance(coeff_of_variation%)"                      
[24] "radiometric_temporal_stability_surface_reflectance_reference_modisbrf(coeff_of_variation%)"   
[25] "radiometric_temporal_stability_surface_reflectance_reference_maiacac(coeff_of_variation%)"    
[26] "radiometric_temporal_stability_gain_surface_reflectance_against_modisbrf(%/year)"             
[27] "radiometric_temporal_stability_gain_surface_reflectance_against_maiacac(%/year)"              
[28] "radiometric_temporal_stability_gain_surface_reflectance_against_modisbrf(coeff_of_variation%)"
[29] "radiometric_temporal_stability_gain_surface_reflectance_against_maiacac(coeff_of_variation%)" 
[30] "acq_id"                                                                                       
[31] "source_file"

In [10]:
colnames(df_abscal)

[1] "evaluation_category"                                               
 [2] "evaluation_type"                                                   
 [3] "affiliation"                                                       
 [4] "constellation"                                                     
 [5] "satellite"                                                         
 [6] "site_name"                                                         
 [7] "product_level"                                                     
 [8] "acq_datetime"                                                      
 [9] "solar_zenith_angle(degree)"                                        
[10] "solar_azimuth_angle(degree)"                                       
[11] "view_zenith_angle(degree)"                                         
[12] "view_azimuth_angle(degree)"                                        
[13] "modis_maiac_aerosol_optical_depth_at_470nm"                        
[14] "modis_maiac_total_column_water_vapor(cm)"                          
[15] "merra2_total_column_ozone(du)"                                     
[16] "band_name"                                                         
[17] "toa_reflectance_at_measured_geometry"                              
[18] "toa_reflectance_at_measured_geometry_reference"                    
[19] "toa_reflectance_at_normalized_view_geometry"                       
[20] "toa_reflectance_at_normalized_view_geometry_reference"             
[21] "gain_toa_reflectance"                                              
[22] "surface_reflectance_at_measured_geometry"                          
[23] "surface_reflectance_at_measured_geometry_reference_modisbrf"       
[24] "surface_reflectance_at_measured_geometry_reference_maiacac"        
[25] "surface_reflectance_at_normalized_view_geometry"                   
[26] "surface_reflectance_at_normalized_view_geometry_reference_modisbrf"
[27] "surface_reflectance_at_normalized_view_geometry_reference_maiacac" 
[28] "gain_surface_reflectance_against_modisbrf"                         
[29] "gain_surface_reflectance_against_maiacac"                          
[30] "acq_id"                                                            
[31] "source_file"

In [11]:
head(df_abscal)

evaluation_category,evaluation_type,affiliation,constellation,satellite,site_name,product_level,acq_datetime,solar_zenith_angle(degree),solar_azimuth_angle(degree),⋯,surface_reflectance_at_measured_geometry,surface_reflectance_at_measured_geometry_reference_modisbrf,surface_reflectance_at_measured_geometry_reference_maiacac,surface_reflectance_at_normalized_view_geometry,surface_reflectance_at_normalized_view_geometry_reference_modisbrf,surface_reflectance_at_normalized_view_geometry_reference_maiacac,gain_surface_reflectance_against_modisbrf,gain_surface_reflectance_against_maiacac,acq_id,source_file
<chr>,<chr>,<chr>,<chr>,<fct>,<chr>,<chr>,<dttm>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
radiometric,abscal,Airbus,Pleiades,PHR1A,PICS Libya-4,Basic,2025-08-22 09:11:32,24.8,128.54,⋯,NaN,0.238,0.246,NaN,0.233,0.241,NaN,NaN,NA,/explore/nobackup/projects/CSDA_eval/results/airbus/pleiades/radiometric/abscal/Radiometric_evaluation_Airbus_Pleiades_abscal_eachcase.csv
radiometric,abscal,Airbus,Pleiades,PHR1A,PICS Libya-4,Basic,2025-08-22 09:11:32,24.8,128.54,⋯,NaN,0.339,0.364,NaN,0.333,0.358,NaN,NaN,NA,/explore/nobackup/projects/CSDA_eval/results/airbus/pleiades/radiometric/abscal/Radiometric_evaluation_Airbus_Pleiades_abscal_eachcase.csv
radiometric,abscal,Airbus,Pleiades,PHR1A,PICS Libya-4,Basic,2025-08-22 09:11:32,24.8,128.54,⋯,NaN,0.479,0.508,NaN,0.471,0.500,NaN,NaN,NA,/explore/nobackup/projects/CSDA_eval/results/airbus/pleiades/radiometric/abscal/Radiometric_evaluation_Airbus_Pleiades_abscal_eachcase.csv
radiometric,abscal,Airbus,Pleiades,PHR1A,PICS Libya-4,Basic,2025-08-22 09:11:32,24.8,128.54,⋯,NaN,0.574,0.646,NaN,0.569,0.640,NaN,NaN,NA,/explore/nobackup/projects/CSDA_eval/results/airbus/pleiades/radiometric/abscal/Radiometric_evaluation_Airbus_Pleiades_abscal_eachcase.csv
radiometric,abscal,Airbus,Pleiades,PHR1A,PICS Libya-4,Basic,2025-09-03 09:18:47,26.4,139.47,⋯,NaN,0.235,0.226,NaN,0.229,0.221,NaN,NaN,NA,/explore/nobackup/projects/CSDA_eval/results/airbus/pleiades/radiometric/abscal/Radiometric_evaluation_Airbus_Pleiades_abscal_eachcase.csv
radiometric,abscal,Airbus,Pleiades,PHR1A,PICS Libya-4,Basic,2025-09-03 09:18:47,26.4,139.47,⋯,NaN,0.335,0.346,NaN,0.329,0.340,NaN,NaN,NA,/explore/nobackup/projects/CSDA_eval/results/airbus/pleiades/radiometric/abscal/Radiometric_evaluation_Airbus_Pleiades_abscal_eachcase.csv


In [12]:
DIR_RADIOMETRIC_DB = file.path(DIR_EVAL_RESULTS_DB, "radiometric_db")
dir.create(DIR_RADIOMETRIC_DB, showWarnings = FALSE, recursive = TRUE)
setwd(DIR_RADIOMETRIC_DB)

In [13]:
write_csv(df_snr, "results_snr.csv")
write_csv(df_abscal, "results_abscal.csv")
write_csv(df_tempstabrad, "results_tempstab.csv")

### Geometric results csvs

In [14]:
# 2. Filter down to only files that live inside an 'apa' subdirectory
apa_files <-         grep("/geometric/apa/.*09012026\\.csv$", all_csvs, value = TRUE)
bbr_files <-         grep("/geometric/bbr/.*09012026\\.csv$", all_csvs, value = TRUE)
ssr_files <-         grep("/geometric/ssr/.*09012026\\.csv$", all_csvs, value = TRUE)
tempstabgeo_files <- grep("/geometric/tempstab/.*09012026\\.csv$", all_csvs, value = TRUE)

In [15]:
# Usage - each file type read separately
df_apa      <- read_evaluation_files(apa_files)
df_ssr      <- read_evaluation_files(ssr_files)
df_bbr      <- read_evaluation_files(bbr_files)
df_tempstabgeo      <- read_evaluation_files(tempstabgeo_files)

New names:
• `` -> `...17`
• `` -> `...18`
• `` -> `...19`
• `` -> `...20`
• `` -> `...21`
• `` -> `...22`


In [16]:
head(df_apa)

evaluation_category,evaluation_type,affiliation,constellation,satellite,site_name,product_level,acq_datetime,acq_id,ref_band,⋯,xoffset_m,yoffset_m,xstd_m,ystd_m,xrmse_m,yrmse_m,ce90_m,ce90-demean_m,band_name,source_file
<chr>,<chr>,<chr>,<chr>,<fct>,<chr>,<chr>,<dttm>,<chr>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<chr>
geometric,apa,Vantor,Legion,1,belo_horizonte,S3DS,2024-10-06,24OCT06130617-S3DS_R5C1-200010869547_01_P001,1,⋯,0.754103,-0.108310,0.570141,1.899454,0.943223,1.890650,1.110162,0.808986,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/apa/LEG-belo_horizonte-apa-09012026.csv
geometric,apa,Vantor,Legion,1,belo_horizonte,S3DS,2024-10-06,24OCT06130617-S3DS_R5C2-200010869547_01_P001,1,⋯,0.801057,0.304296,0.349616,0.183214,0.872861,0.354406,1.150507,0.409957,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/apa/LEG-belo_horizonte-apa-09012026.csv
geometric,apa,Vantor,Legion,1,belo_horizonte,S3DS,2024-10-06,24OCT06130617-S3DS_R6C1-200010869547_01_P001,1,⋯,0.844994,-0.062115,0.307379,0.217993,0.898918,0.226177,1.230074,0.598724,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/apa/LEG-belo_horizonte-apa-09012026.csv
geometric,apa,Vantor,Legion,1,belo_horizonte,S3DS,2024-10-06,24OCT06130617-S3DS_R6C2-200010869547_01_P001,1,⋯,0.831803,0.377818,0.220860,0.182972,0.860461,0.419560,1.245892,0.444095,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/apa/LEG-belo_horizonte-apa-09012026.csv
geometric,apa,Vantor,Legion,1,belo_horizonte,S3DS,2024-10-06,24OCT06130617-S3DS_R7C1-200010869547_01_P001,1,⋯,0.945622,0.085211,0.228758,0.307854,0.972753,0.318626,1.307971,0.576325,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/apa/LEG-belo_horizonte-apa-09012026.csv
geometric,apa,Vantor,Legion,1,belo_horizonte,S3DS,2024-10-06,24OCT06130617-S3DS_R7C2-200010869547_01_P001,1,⋯,1.087873,0.530414,0.728809,0.395686,1.308068,0.660945,1.993111,1.081225,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/apa/LEG-belo_horizonte-apa-09012026.csv


In [17]:
colnames(df_bbr)

[1] "evaluation_category" "evaluation_type"     "affiliation"        
 [4] "constellation"       "satellite"           "site_name"          
 [7] "product_level"       "acq_datetime"        "ref_band"           
[10] "test_band"           "n_chips"             "xoffset_m"          
[13] "yoffset_m"           "xstd_m"              "ystd_m"             
[16] "xrmse_m"             "yrmse_m"             "ce90_m"             
[19] "ce90-demean_m"       "acq_id"              "band_name"          
[22] "source_file"

In [18]:
DIR_GEOMETRIC_DB = file.path(DIR_EVAL_RESULTS_DB, "geometric_db")
dir.create(DIR_GEOMETRIC_DB, showWarnings = FALSE, recursive = TRUE)
setwd(DIR_GEOMETRIC_DB)

In [19]:
write_csv(df_apa, "results_apa.csv")
write_csv(df_ssr, "results_ssr.csv")
write_csv(df_bbr, "results_bbr.csv")
write_csv(df_tempstabgeo, "results_tempstab.csv")

### Create acquisitions table

In [20]:
# ============================================================
# COMMON COLS FOR ACQUISITIONS TABLE
# ============================================================
common_cols <- c(
  'evaluation_category','evaluation_type', 'affiliation', 'constellation', 'satellite',
  'site_name', 'product_level',
  'acq_datetime', 
    #'date', 
    'source_file'
)

# ============================================================
# BUILD ACQUISITIONS TABLE
# Collapse band rows -> one row per unique Acq_Datetime
# (not uding TempStab)
# ============================================================
acquisitions <- bind_rows(
    # Radiometric dB
    df_snr %>% select(any_of(common_cols), band_name),
    df_abscal %>% select(any_of(common_cols), band_name),
    # Geometric dB
    df_apa %>% select(any_of(common_cols), band_name),
    df_ssr %>% select(any_of(common_cols), band_name),
    df_bbr %>% select(any_of(common_cols), band_name),
  ) %>%
  group_by(across(all_of(common_cols))) %>%
  summarise(
    bands      = list(sort(unique(as.character(band_name)))),  # Collapsed band list
    n_bands    = n_distinct(band_name),
    .groups    = 'drop'
  ) %>%
  # Unique acquisitions by Acq_Datetime
  distinct(acq_datetime, .keep_all = TRUE) %>%
  arrange(acq_datetime) %>%
  relocate(source_file, .after = everything()) %>%
  #mutate(acq_id = row_number()) %>%
  #relocate(acq_id, .before = everything()) %>%
  #mutate(Evaluation_category = 'Radiometric') %>%
  #relocate(Evaluation_category, .before = everything()) %>%
  as.data.frame()

setwd(DIR_EVAL_RESULTS_DB)
write_csv(acquisitions, 'acquisitions.csv')

### Make readable

In [21]:
system(paste("chmod -R 755", shQuote(DIR_EVAL_RESULTS_DB)))

In [22]:
# Preview
acquisitions %>%
  select(
      #acq_id, 
      'affiliation', 'constellation', 'satellite', acq_datetime, n_bands, bands) %>%
  head(3)

print(glue::glue("CSDA Evaluation Acquisitions: {nrow(acquisitions)} unique acquisitions"))

,affiliation,constellation,satellite,acq_datetime,n_bands,bands
,<chr>,<chr>,<fct>,<dttm>,<int>,<list>
1,Satellogic,Newsat,SN10,2021-02-02 09:01:41,4,"Blue , Green, NIR , Red"
2,Satellogic,Newsat,SN10,2021-02-05 09:10:47,4,"Blue , Green, NIR , Red"
3,Satellogic,Newsat,SN10,2021-02-12 09:00:48,4,"Blue , Green, NIR , Red"


CSDA Evaluation Acquisitions: 104 unique acquisitions


In [23]:
# TODO: need to change source bbr tables to be site specific 
tail(acquisitions %>% filter(site_name == 'all_geometric'))

,evaluation_category,evaluation_type,affiliation,constellation,satellite,site_name,product_level,acq_datetime,bands,n_bands,source_file
,<chr>,<chr>,<chr>,<chr>,<fct>,<chr>,<chr>,<dttm>,<list>,<int>,<chr>
1,geometric,bbr,Vantor,Legion,NA,all_geometric,M3DS,NA,,1,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/bbr/bbr_8band_LEG-09012026.csv


In [24]:
tail(acquisitions)

,evaluation_category,evaluation_type,affiliation,constellation,satellite,site_name,product_level,acq_datetime,bands,n_bands,source_file
,<chr>,<chr>,<chr>,<chr>,<fct>,<chr>,<chr>,<dttm>,<list>,<int>,<chr>
99,geometric,apa,Vantor,Legion,5,melbourne,S3DS,2025-11-25 00:00:00,,1,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/apa/LEG-melbourne-apa-09012026.csv
100,radiometric,abscal,Airbus,Pleiadesneo,PNEO3,PICS Libya-4,Basic and Surface reflectance,2026-02-25 09:07:49,"Blue , Coastal, Green , NIR , Red , RedEdge",6,/explore/nobackup/projects/CSDA_eval/results/airbus/pleiadesneo/radiometric/abscal/Radiometric_evaluation_Airbus_PleiadesNeo_abscal_eachcase.csv
101,radiometric,abscal,Airbus,Pleiades,PHR1A,PICS Libya-4,Basic,2026-02-25 09:21:42,"Blue , Green, NIR , Red",4,/explore/nobackup/projects/CSDA_eval/results/airbus/pleiades/radiometric/abscal/Radiometric_evaluation_Airbus_Pleiades_abscal_eachcase.csv
102,radiometric,abscal,Airbus,Spot,SPOT6,PICS Libya-4,Basic,2026-02-26 08:46:49,"Blue , Green, NIR , Red",4,/explore/nobackup/projects/CSDA_eval/results/airbus/spot6/radiometric/abscal/Radiometric_evaluation_Airbus_SPOT_abscal_eachcase.csv
103,radiometric,abscal,Airbus,Pleiadesneo,PNEO4,PICS Libya-4,Basic and Surface reflectance,2026-03-05 09:11:35,"Blue , Coastal, Green , NIR , Red , RedEdge",6,/explore/nobackup/projects/CSDA_eval/results/airbus/pleiadesneo/radiometric/abscal/Radiometric_evaluation_Airbus_PleiadesNeo_abscal_eachcase.csv
104,geometric,bbr,Vantor,Legion,NA,all_geometric,M3DS,NA,,1,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/bbr/bbr_8band_LEG-09012026.csv


In [25]:
# # ============================================================
# # BUILD RESULTS TABLES (evaluation-specific metrics)
# # ============================================================

# # Helper to add acq_id by joining on common cols
# add_acq_id <- function(df){
#   df %>%
#     left_join(acquisitions %>% select(acq_id, all_of(common_cols)),
#               by = common_cols) %>%
#     relocate(acq_id, .before = everything())
# }

# # SNR results
# results_snr <- df_snr %>%
#   select(-any_of(common_cols)) %>%
#   bind_cols(df_snr %>% select(all_of(common_cols))) %>%
#   add_acq_id() %>%
#   select(acq_id, #any_of(angular_atm_cols), 
#          Signal_to_noise_ratio,
#          TOA_reflectance_at_normalized_view_geometry)

# # AbsCal results
# results_abscal <- df_abs %>%
#   select(-any_of(common_cols)) %>%
#   bind_cols(df_abs %>% select(all_of(common_cols))) %>%
#   add_acq_id() %>%
#   select(acq_id, #any_of(angular_atm_cols),
#          TOA_reflectance_at_normalized_view_geometry,
#          TOA_reflectance_at_normalized_view_geometry_reference,
#          Gain_TOA_reflectance,
#          Surface_reflectance,
#          Surface_reflectance_reference1,
#          Surface_reflectance_reference2,
#          Gain_surface_reflectance_1,
#          Gain_surface_reflectance_2)

# # TempStab results
# results_tempstab <- df_tempstab %>%
#   select(-any_of(common_cols)) %>%
#   bind_cols(df_tempstab %>% select(all_of(common_cols))) %>%
#   add_acq_id() %>%
#   select(acq_id,
#          Time_series_date_min,
#          Time_series_date_max,
#          Number_of_images,
#          `Time_series_length(days)`,
#          starts_with('Radiometric_Temporal_Stability'))

# # # ============================================================
# # # SAVE TO SQLITE DATABASE
# # # ============================================================
# # db_path <- 'radiometric_eval_db.sqlite'
# # con <- dbConnect(SQLite(), db_path)

# # # Write tables
# # dbWriteTable(con, 'acquisitions',    acquisitions,    overwrite = TRUE)
# # dbWriteTable(con, 'results_snr',     results_snr,     overwrite = TRUE)
# # dbWriteTable(con, 'results_abscal',  results_abscal,  overwrite = TRUE)
# # dbWriteTable(con, 'results_tempstab',results_tempstab, overwrite = TRUE)

# # # Add indexes for fast joins
# # dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_acq_id_snr      ON results_snr(acq_id)')
# # dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_acq_id_abscal   ON results_abscal(acq_id)')
# # dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_acq_id_tempstab ON results_tempstab(acq_id)')
# # dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_satellite        ON acquisitions(Satellite)')
# # dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_eval_type        ON acquisitions(Evaluation_type)')

# # dbDisconnect(con)

# # print(glue::glue("Database saved to: {db_path}"))
# # print(glue::glue("Tables: acquisitions ({nrow(acquisitions)}), 
# #                   SNR ({nrow(results_snr)}), 
# #                   AbsCal ({nrow(results_abscal)}), 
# #                   TempStab ({nrow(results_tempstab)})"))